In [1]:
"""
This script tests the dimensions of a Pioreactor on the Starlet Deck. 
The pioreactor sits on a plate adapter and is calculated as part of the Pioreactor height. 
This script works correctly. 20250918
Next steps:
create a new git branch.
Add the Hamilton 3d print supports.
Creat another new branch. How does this inherit 3d print changes from branch and not main?
Add the Pioreactor resource.
Test the imports with both branches.
Write script that samples every 30 min.
"""


'\nThis script tests the dimensions of a Pioreactor on the Starlet Deck. \nThe pioreactor sits on a plate adapter and is calculated as part of the Pioreactor height. \nThis script works correctly. 20250918\nNext steps:\ncreate a new git branch.\nAdd the Hamilton 3d print supports.\nCreat another new branch. How does this inherit 3d print changes from branch and not main?\nAdd the Pioreactor resource.\nTest the imports with both branches.\nWrite script that samples every 30 min.\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources import Coordinate
# from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
# from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
# from pylabrobot.resources.pioreactor.bioreactor import pioreactor

from pylabrobot.resources import (
    TIP_50ul_w_filter, # 50 µL filtered
    HTF,     # 1000 µL filtered 
    LTF #Tip Rack with 96 10ul Low Volume Tip with filter
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

In [4]:
from pylabrobot.resources.carrier import Coordinate, PlateHolder
from pylabrobot.resources.resource_holder import ResourceHolder
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint


# def Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint(name: str) -> PlateHolder:
#   """Hamilton MFX DWP Module (cat.-no. 188042 / 188042-00).
#   It also contains metal clamps at the corners.
#   https://www.hamiltoncompany.com/other-robotics/188042
#   """

#   return PlateHolder(
#     name=name,
#     size_x=135.0,  # measured
#     size_y=94.0,  # measured
#     size_z=29.8,  # measured
#     # probe height - carrier_height - deck_height
#     child_location=Coordinate(4.0, 4.0, 183.95 - 63.95 - 100),  # 20.0 measured
#     pedestal_size_z=-4.74,
#     model=Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint.__name__,
#   )

In [5]:

# original simplified below
# """
# Pioreactor 'plate' resource for PyLabRobot.

# This models the single-vessel Pioreactor as a *skirted 1×1 Plate* so it can be
# assigned to a PlateHolder (e.g. Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint).

# Geometry (mm)
# -------------
# - External footprint on holder: X=127.74, Y=85.4
# - Central vial (A1 well): inner Ø = 23.5, outer Ø = 27.5, depth = 57
# - Plate Z (overall height used by PLR for collisions): well depth + plastic
#   thickness (default plastic thickness = 1.0 mm)

# Usage
# -----
# >>> from pylabrobot.resources import Coordinate
# >>> pr = pioreactor("pr1")
# >>> holder.assign_child_resource(pr)          # PlateHolder accepts Plates
# >>> await lh.aspirate(pr["A1"], vols=200,
# ...                   offsets=Coordinate(0, 0, 2.0))  # +2 mm above bottom
# """

# from __future__ import annotations

# from dataclasses import dataclass

# # ---- Imports with version fallbacks -----------------------------------------
# try:
#   # PLR ≥ 0.10 (preferred)
#   from pylabrobot.resources import Plate, Coordinate
#   from pylabrobot.resources.well import Well, WellBottomType, CrossSectionType
#   try:
#     # Newer helper
#     from pylabrobot.resources.utils import create_ordered_items_2d as _mk_items
#   except Exception:
#     # Older helper name
#     from pylabrobot.resources.utils import create_equally_spaced_2d as _mk_items
# except Exception:  # pragma: no cover - older PLR fallback
#   from pylabrobot.resources.plate import Plate
#   from pylabrobot.resources.coordinate import Coordinate
#   from pylabrobot.resources.well import Well, WellBottomType, CrossSectionType
#   try:
#     from pylabrobot.resources.utils import create_ordered_items_2d as _mk_items
#   except Exception:
#     from pylabrobot.resources.utils import create_equally_spaced_2d as _mk_items


# # ---- Physical constants (mm) -------------------------------------------------
# PIOREACTOR_SIZE_X = 127.74 # measured
# PIOREACTOR_SIZE_Y = 85.40 # measured
# # PIOREACTOR_SIZE_Z = 126.0  # measured
# PIOREACTOR_SIZE_Z = 140.0  # measured

# VIAL_INNER_DIAMETER = 23.5 # spec
# VIAL_OUTER_DIAMETER = 27.5 # spec
# # VIAL_DEPTH = 57.0 # lower means tip doesn't go as far down
# VIAL_DEPTH = 57.0 # spec

# # thickness of plastic between holder deck and start of cavity (≈ "dz" in PLR)
# DEFAULT_MATERIAL_Z_THICKNESS = 1.0 # measured


# @dataclass(frozen=True)
# class VialSpec:
#   inner_d: float = VIAL_INNER_DIAMETER
#   outer_d: float = VIAL_OUTER_DIAMETER
#   depth: float = VIAL_DEPTH


# class PioreactorPlate(Plate):
#   """Pioreactor as a skirted 1×1 Plate with a single circular well (A1)."""

#   def __init__(
#     self,
#     name: str,
#     *,
#     model: str | None = None,
#     vial: VialSpec | None = None,
#     material_z_thickness: float = DEFAULT_MATERIAL_Z_THICKNESS,
#     plate_type: str = "skirted",   # PlateHolder expects skirted plates
#   ):
#     vial = vial or VialSpec()

#     # Center the single circular well inside the external footprint
#     well_d = vial.inner_d
#     dx = (PIOREACTOR_SIZE_X - well_d) / 2.0  # left margin to well
#     dy = (PIOREACTOR_SIZE_Y - well_d) / 2.0  # front margin to well
#     dz = PIOREACTOR_SIZE_Z - vial.depth   # increase means farther up vial; decrease is towards deck

#     # Build a 1×1 grid of Wells (A1 only), circular cross-section, flat bottom.
#     ordered_items = _mk_items(
#       Well,
#       num_items_x=1,
#       num_items_y=1,
#       dx=dx,
#       dy=dy,
#       dz=dz,
#       item_dx=0.0,
#       item_dy=0.0,
#       size_x=well_d,          # diameter in x for CIRCLE
#       size_y=well_d,          # diameter in y for CIRCLE
#       size_z=vial.depth,      # cavity depth
#       bottom_type=WellBottomType.FLAT,
#       cross_section_type=CrossSectionType.CIRCLE,
#       material_z_thickness=material_z_thickness,
#     )

#     # Plate overall height = cavity depth + bottom plastic thickness
#     # plate_size_z = vial.depth + material_z_thickness
#     plate_size_z = PIOREACTOR_SIZE_Z

#     # Plate ctor signature changed over time ("ordered_items" vs "items").
#     try:
#       super().__init__(
#         name=name,
#         size_x=PIOREACTOR_SIZE_X,
#         size_y=PIOREACTOR_SIZE_Y,
#         size_z=plate_size_z,
#         lid=None,
#         model=model or self.__class__.__name__,
#         ordered_items=ordered_items,
#       )
#     except TypeError:
#       super().__init__(
#         name=name,
#         size_x=PIOREACTOR_SIZE_X,
#         size_y=PIOREACTOR_SIZE_Y,
#         size_z=plate_size_z,
#         lid=None,
#         model=model or self.__class__.__name__,
#         items=ordered_items,
#       )

#     # Some PlateHolders explicitly check this attribute.
#     self.plate_type = plate_type

#   # ---- convenience -----------------------------------------------------------
#   @property
#   def A1(self) -> Well:
#     """Return the single well (alias)."""
#     return self["A1"]

#   def recommended_aspirate_offset(self) -> Coordinate:
#     """Conservative default: +2 mm above bottom center of the well."""
#     return Coordinate(0.0, 0.0, 2.0)


# # Factory for backwards compatibility with your earlier code -------------------
# def pioreactor(name: str) -> PioreactorPlate:
#   """Create a Pioreactor as a 1×1 Plate (skirted)."""
#   return PioreactorPlate(name=name)


In [6]:
###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01") #  50 µL filter tips (slot-1)
tiprack_10 = LTF("tips_02") #10 ul filter tips
# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10


In [7]:
# import a DW plate
from pylabrobot.resources.bioer import BioER_96_wellplate_Vb_2200ul
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
dwplate = BioER_96_wellplate_Vb_2200ul('dwPlate')
plate0 = Hamilton_MFX_plateholder_DWP_metal_tapped('plate0')
car_13 = MFX_CAR_L5_base('car_13', modules={0:plate0})
lh.deck.assign_child_resource(car_13, rails=13)
plate0.assign_child_resource(dwplate)


In [8]:


# # STANDARDS RACK
holder   = Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("mfx_mod_1")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: holder
    }
)
lh.deck.assign_child_resource(car_07, rails=7)




In [ ]:
from pylabrobot.resources.pioreactor import pioreactor

# simplified 
# from typing import Optional

# from pylabrobot.resources import Plate, Lid
# from pylabrobot.resources.utils import create_ordered_items_2d
# from pylabrobot.resources.well import Well, WellBottomType, CrossSectionType

# def pioreactor(name: str, lid: Optional[Lid] = None) -> Plate:
#   """
#   Pioreactor 20mL Vessel: https://pioreactor.com/products/pioreactor-20ml
#   Modeled as a 1x1 skirted plate

#   Geometry (mm):
#   - Outer footprint (on holder): 127.74 x 85.40
#   - Total height (plate Z): 126.5
#   - Central vial (A1): inner Ø 23.5, depth 57.0
#   """

#   # --- Outer dimensions (measured) ---
#   OUTER_X = 127.74
#   OUTER_Y = 85.40
#   OUTER_Z = 126.5  # overall height used for collision checks

#   # --- Well (vial) geometry (spec/measured) ---
#   WELL_DIAMETER = 23.5
#   WELL_DEPTH = 57.0
#   MATERIAL_Z_THICKNESS = 1.0  # plastic between top surface and cavity start

#   # Center the single circular well
#   dx = (OUTER_X - WELL_DIAMETER) / 2.0
#   dy = (OUTER_Y - WELL_DIAMETER) / 2.0

#   # Distance from plate top to top of cavity
#   # dz = OUTER_Z - WELL_DEPTH - MATERIAL_Z_THICKNESS
#   # dz = 126.5-57 -1 =68.5 # tip crashes into bottom
#   dz = 76 # measured

#   # Cylinder area for volume/height conversions
#   cross_section_area = 3.14 * (WELL_DIAMETER / 2.0) ** 2

#   well_kwargs = {
#     "size_x": WELL_DIAMETER,               # for CIRCLE, size_x == size_y == diameter
#     "size_y": WELL_DIAMETER,
#     "size_z": WELL_DEPTH,
#     "bottom_type": WellBottomType.FLAT,
#     "cross_section_type": CrossSectionType.CIRCLE,
#     "compute_height_from_volume": lambda v: v / cross_section_area,
#     "compute_volume_from_height": lambda h: h * cross_section_area,
#     "material_z_thickness": MATERIAL_Z_THICKNESS,
#   }

#   return Plate(
#     name=name,
#     size_x=OUTER_X,
#     size_y=OUTER_Y,
#     size_z=OUTER_Z,
#     lid=lid,
#     model="PioreactorPlate",
#     ordered_items=create_ordered_items_2d(
#       Well,
#       num_items_x=1,
#       num_items_y=1,
#       dx=dx,
#       dy=dy,
#       dz=dz,
#       item_dx=WELL_DIAMETER,
#       item_dy=WELL_DIAMETER,
#       **well_kwargs,
#     ),
#   )


In [10]:
pr1 = pioreactor("pr1")
holder.assign_child_resource(pr1)  

Resource 'pr1' is very high on the deck: 259.955 mm. Be careful when traversing the deck.
Resource 'pr1_well_0_0' is very high on the deck: 266.455 mm. Be careful when traversing the deck.


In [11]:
CHANNEL_MM   = 6            # single channel we’ll use for the whole run
TIPRACK_50   = tiprack_50   # 50 µL filter tips (slot-1 on tip_car)
SAFE_Z = 270  # mm above deck; set > tallest stack by ~5–10 mm
# st = 10

await lh.pick_up_tips(TIPRACK_50["A1"], use_channels=[CHANNEL_MM])

await lh.aspirate(
  pr1["A1"], vols=[0],
  use_channels=[CHANNEL_MM], settling_time=[9],
  minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
  min_z_endpos=SAFE_Z
)

await lh.dispense(
  dwplate["A1"], vols=[0],
  use_channels=[CHANNEL_MM], settling_time=[1],
  minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
  min_z_endpos=SAFE_Z
)

In [12]:

# CHANNEL_MM   = 6            # single channel we’ll use for the whole run
# TIPRACK_50   = tiprack_50   # 50 µL filter tips (slot-1 on tip_car)
# # st = 10

# await lh.pick_up_tips(TIPRACK_50["B1"], use_channels=[CHANNEL_MM])


# await lh.aspirate(pr1["A1"], vols=[2], use_channels=[CHANNEL_MM], settling_time=[9])
# await lh.dispense(dwplate["A1"], vols=[2], use_channels=[CHANNEL_MM], liquid_height=[2], settling_time=[1])



In [13]:
# from pylabrobot.resources import Coordinate

# # 1) get the single Well object
# well = pr1.get_well("A1") if hasattr(pr1, "get_well") else pr1["A1"][0]

# # 2) absolute XY of vial center at the **bottom** of the cavity
# p_btm = well.get_absolute_location(x="c", y="c", z="b")  # absolute coords

# # 3) compute target Z = bottom + vial depth + clearance
# depth = getattr(well, "depth", None) or getattr(well, "size_z", None) or getattr(pr1, "vial_height_mm", None) or 57.0
# CLEAR = 5.0
# p_target = p_btm + Coordinate(0, 0, depth + CLEAR)  # rim + CLEAR

# # 4) move the selected channel there, safely
# channel = CHANNEL_MM
# await lh.prepare_for_manual_channel_operation(channel)

# z_safe = max(lh.deck.get_highest_known_point() + 10.0, p_target.z + 20.0)  # play it safe
# await lh.move_channel_z(channel, z_safe)          # up above everything
# await lh.move_channel_x(channel, p_target.x)      # traverse in X
# await lh.move_channel_y(channel, p_target.y)      # traverse in Y
# await lh.move_channel_z(channel, p_target.z)      # come down to top-of-vial + CLEAR


In [14]:
# pr16 = pioreactor("pr16")
# holder.assign_child_resource(pr16)
# # avoids 15mm crossbar stirbar
# await lh.aspirate(
#     pr16["A1"],
#     vols=[0],
#     use_channels=[CHANNEL_MM],
#     offsets=[Coordinate(z=8.0)],
#     settling_time=[9])
# await lh.move_channel_z(CHANNEL_MM, 270)      # come down to top-of-vial + CLEAR

In [15]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[100], liquid_height=[4], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.dispense(pr1["A1"], vols=[50], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.drop_tips(TIPRACK_50["A1"], use_channels=[6])
# await lh.discard_tips()
# await lh.stop()
# await backend.stop() 